Building likelihood maps for a set of representative images.

> TODO: use images from dataset and superimpose the bounding boxes on them

In [1]:
from retinotopy import *
welcome()

-----------------------------------------------------------------------------------------
On date 2025-03-06, Running learning on host obiwan.local with device mps, pytorch==2.6.0
-----------------------------------------------------------------------------------------
Welcome on macOS-15.3.2-arm64-arm-64bit


In [2]:
# The dataset to import images from

data_set_type = 'full'

args = Params()
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
folder = args.folders[0]
args.folders = ['val'] # type of images to use


args.do_saccade = False
args.do_resize = False
args.do_mask = False
args.do_polar = False

image_dataset = image_datasets_transforms(args, verbose=False)['val']
N_image = 1000
np.random.seed(args.seed)
idxs = np.random.permutation(len(image_dataset))[:N_image]

args = Params()
data_transforms = {}
args.do_saccade = True
args.normalize = False
args.device = device
args.size_ratio = 0.5
args.method = 'full'
args.saccade_type = 'grid'
args.resolution = (11,11)

for do_polar in [False, True]:
    args.do_polar = do_polar
    data_transforms[do_polar] = get_transforms(args)

N_image = 3
np.random.seed(args.seed)
idxs = np.random.permutation(len(image_dataset))[:N_image]

do_save = False

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/SSD1TO/Deep_learning/data/Imagenet_full/val'

## doing the computations

In [ ]:
image_names = ['jaguar', 'jaguar_5', 'cheetah']
true_labels = ['jaguar', 'leopard', 'cheetah']

In [ ]:
for model_data_set_type in data_set_types:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:

        print(50*'.')
        model = {}
        for do_polar in [False, True]:
            do_polar = do_polar if data_set_type != 'raw' else False
            model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + '.pt' if model_data_set_type != 'raw' else None
            model[do_polar] = load_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
                
        pprint(f'Testing : {model_name}')

        # torch.cuda.amp.GradScaler(enabled=True)

        for image_name, true_label in zip(image_names, true_labels):
            map_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + f'_map_{image_name}.npy'
            
            image_url = f'./imgs/{image_name}.jpg'
            full_image = read_image(image_url)/255
            
            full_image_np = torch.movedim(full_image, (1, 2, 0), (0, 1, 2)).numpy()

            pos_H, pos_W, box_size = get_positions(args, full_image)
    
            fig, axs = plt.subplots(1, 3, figsize=(fig_width*2, fig_width/2))
            three, H, W = full_image.shape
            ax = axs[0]
            extent = (0, W, 0, H)
            ax.imshow(full_image_np, origin="upper", extent=extent)
            ax.set_xticks([])
            ax.set_yticks([])
                
            full_image = full_image.to(device, non_blocking=True)
            
            for i_polar, do_polar in enumerate([False, True]):
                fixation_map = data_transforms[do_polar](full_image)
                proba_label = get_batch(args, model[do_polar], fixation_map, 7)
                proba_label = proba_label[:, i_labels_dico[true_label]].reshape(args.resolution)
                
                if do_save:
                    np.save(map_filename, proba_label)

                if False:
                    if os.path.isfile(map_filename):
                        proba_label = np.load(map_filename)
                
                proba_max = 1.
                mycmap = transparent_cmap(plt.cm.Blues if i_polar == 0 else plt.cm.Reds)
                ax = axs[i_polar+1]
                ax.contourf(pos_W, H-pos_H, proba_label, levels=np.linspace(0, proba_max, 10), origin="upper", extent=extent, cmap=mycmap, vmin=0, vmax=proba_max)
                ax.set_xticks([])
                ax.set_yticks([])

        
        fig.set_facecolor(color='white')
        plt.show()
    


## doing the figure

In [ ]:
if do_save:
    for data_set_type in data_set_types:
        print(50*'=')
        print(f'{data_set_type=}')    
        for model_name in  ['resnet18', 'resnet50', 'resnet101']:
            print(f'{model_name=}')    
            print(50*'.')
    
            fig, axs = plt.subplots(len(image_names), 3, figsize=(fig_width*len(image_names), fig_width))
            for i_image, (image_name, true_label) in enumerate(zip(image_names, true_labels)):
    
                image_url = f'./imgs/{image_name}.jpg'
                full_image = read_image(image_url)/255
                full_image_np = torch.movedim(full_image, (1, 2, 0), (0, 1, 2)).numpy()
                three, H, W = full_image.shape
    
                ax = axs[i_image, 0]
                extent = (0, W, 0, H)
                ax.imshow(full_image_np, origin="upper", extent=extent)
                ax.set_xticks([])
                ax.set_yticks([])
    
                # ax.set_xticks([])
                pos_H, pos_W, box_size = get_positions(full_image, resolution, size_ratio)
                resolution = pos_H.shape
                for i_polar, do_polar in enumerate([False, True]):
                    map_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + f'_map_{image_name}.npy'
                    if os.path.isfile(map_filename):
                        proba_label = np.load(map_filename)
                        proba_max = proba_label.max()
                        proba_max = 1.
                        mycmap = transparent_cmap(plt.cm.Reds if do_polar else plt.cm.Blues)
                        ax = axs[i_image,  i_polar+1]
                        ax.contourf(pos_W, H-pos_H, proba_label, levels=np.linspace(0, proba_max, 10), origin="upper", extent=extent, cmap=mycmap, vmin=0, vmax=proba_max, linewidths=3)
                        ax.set_xticks([])
                        ax.set_yticks([])
            # ax.scatter(500, 100, s=1000, c='r')
            # ax.set_yticks([])  
            fig.set_facecolor(color='white')
            plt.show()
    
if do_save: to_save(fig, name='fig-likelihood_map')


In [ ]:
retino_grid = get_grid(args).unsqueeze(0)
for model_data_set_type in data_set_types:
    
    print(50*'=')
    print(f'{model_data_set_type=}')    

    for model_name in  ['resnet18', 'resnet50', 'resnet101']:

        print(50*'.')
        model = {}
        for do_polar in [False, True]:
            do_polar = do_polar if data_set_type != 'raw' else False
            model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + '.pt' if model_data_set_type != 'raw' else None
            model[do_polar] = load_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
                
        pprint(f'Testing : {model_name}')

        # torch.cuda.amp.GradScaler(enabled=True)
        
        for i_image, ind in enumerate([1998]):
            
            (full_image, true_label) = image_dataset[ind]
            full_image_np  = imgs_to_np(full_image)

            pos_H, pos_W, box_size = get_positions(args, full_image)
            fig, axs = plt.subplots(1, 3, figsize=(fig_width*2, fig_width/2))
            three, H, W = full_image.shape
    
            ax = axs[0]
            extent = (0, W, 0, H)
            ax.imshow(full_image_np, origin="upper", extent=extent)
            ax.set_xticks([])
            ax.set_yticks([])

            full_image = full_image.to(device, non_blocking=True)
            
            for i_polar, do_polar in enumerate([False, True]):
                #fixation_map = data_transforms[do_polar](full_image)
                fixation_map = rolling_map_LP(args, (H, W), full_image, retino_grid)
                proba_label = get_batch(args, model[do_polar], fixation_map, 7)
                proba_label = proba_label[:, true_label].reshape(args.resolution)
                
                if do_save:
                    np.save(map_filename, proba_label)

                proba_max = 1.
                mycmap = transparent_cmap(plt.cm.Blues if i_polar == 0 else plt.cm.Reds)
                ax = axs[i_polar+1]
                ax.contourf(pos_W, H-pos_H, proba_label, levels=np.linspace(0, proba_max, 10), origin="upper", extent=extent, cmap=mycmap, vmin=0, vmax=proba_max)
                ax.set_xticks([])
                ax.set_yticks([])

        fig.set_facecolor(color='white')
        plt.show()
    


In [ ]:
retino_grid = get_grid(args).unsqueeze(0)
model_raw = load_model(model_name='resnet101', model_path=None, do_circular=False).to(device).eval()
print(args)
num_image_sample = 5
set_seed(seed=6806)
for model_data_set_type in ['bbox']:
    
    print(50*'=')
    print(f'{model_data_set_type=}')    

    for model_name in  ['resnet101']:

        print(50*'.')
        model = {}
        for do_polar in [False, True]:
            model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + '.pt'
            model[do_polar] = load_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
                
        pprint(f'Testing : {model_name}')

        # torch.cuda.amp.GradScaler(enabled=True)
        fig, all_axs = plt.subplots(num_image_sample, 4, figsize=(fig_width, fig_width*2), squeeze=True)
        for i_image, ind in enumerate((np.random.rand(num_image_sample)*50000)):
            ind = int(ind)
            axs = all_axs[i_image]
            (full_image, true_label) = image_dataset[ind]
            full_image_np  = imgs_to_np(full_image)
            
            
            print(labels[true_label], ind)
            labels_i_image = labels[true_label]

    
            ax = axs[0]
            extent = (0, full_image.shape[1], 0, full_image.shape[2])

            ax.imshow(full_image_np, origin="upper", extent=extent)
            ax.set_ylabel(labels_i_image)

            ax.set_xticks([])
            ax.set_yticks([])

            full_image = full_image.to(device, non_blocking=True)
            
            for i_polar, do_polar in enumerate([False, True]): 
                
                args.do_polar = do_polar
                fixation_map = data_transforms[do_polar](full_image)

                proba_label = get_batch(args, model[do_polar], fixation_map, 50)
                proba_label = proba_label[:, true_label].reshape(args.resolution)

                ax = axs[i_polar+2]
                
                display_heat(full_image_np, proba_label, args.resolution, model_name,  ax)
                
                ax.set_xticks([])
                ax.set_yticks([])
                
                if not do_polar:
                    args.do_polar = False
                    
                    fixation_map = data_transforms[False](full_image)
        
                    proba_label = get_batch(args, model_raw, fixation_map, 50)
                    proba_label = proba_label[:, true_label].reshape(args.resolution)
                    ax = axs[1]
                    display_heat(full_image_np, proba_label, args.resolution, model_name,  ax)
                   

        all_axs[0,0].set_title('Image')
        all_axs[0,1].set_title("No re-train")
        all_axs[0,2].set_title("Cartesian")
        all_axs[0,3].set_title("Retino")
        fig.set_facecolor(color='white')
        plt.show()
    
    

In [ ]:
def display_heat(image, likelihood_map, resolution, model_name, ax):
    shape_im = image.shape[:2]
    likelihood_map = np.array(likelihood_map).reshape(resolution)
    likelihood_map = cv2.resize(np.array(likelihood_map), (shape_im[1],shape_im[0]), interpolation=cv2.INTER_NEAREST )
    #likelihood_map = cv2.resize(np.array(likelihood_map), (shape_im[1],shape_im[0]), interpolation=None)
    image_lin_display = cv2.resize(image, likelihood_map.T.shape, interpolation= cv2.INTER_LINEAR)
    sns.heatmap(likelihood_map, linewidth = 0 , annot = False, cmap=cmap, vmin=0, vmax=1, cbar = False, alpha=.5, ax=ax)
    ax.imshow(image_lin_display)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()


In [ ]:
retino_grid = get_grid(args).unsqueeze(0)
model_raw = load_model(model_name='resnet101', model_path=None, do_circular=False).to(device).eval()
print(args)
num_image_sample = 4
sample_dis = [1998, 45706, 945, 26415]
labels_dis = ['Iguana', 'Yawl', 'Magpie', 'Dial telephone']
#np.random.seed(23613)
for model_data_set_type in ['bbox']:
    
    print(50*'=')
    print(f'{model_data_set_type=}')    

    for model_name in  ['resnet101']:

        print(50*'.')
        model = {}
        for do_polar in [False, True]:
            args.do_polar = do_polar
            model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + '.pt'
            model[do_polar] = load_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
                    
        pprint(f'Testing : {model_name}')

        # torch.cuda.amp.GradScaler(enabled=True)
        fig, all_axs = plt.subplots(num_image_sample, 4, figsize=(15, 15), squeeze=True)
        for i_image, ind in enumerate(sample_dis): 
            ind = int(ind)
            axs = all_axs[i_image]
            (full_image, true_label) = image_dataset[ind]
            full_image_np  = imgs_to_np(full_image)
            
            
            print(labels[true_label], ind)
            labels_i_image = labels[true_label]

    
            ax = axs[0]
            extent = (0, full_image.shape[1], 0, full_image.shape[2])

            ax.imshow(full_image_np)#, origin="upper", extent=extent)
            ax.set_ylabel(labels_dis[i_image] , fontsize=25)

            ax.set_xticks([])
            ax.set_yticks([])

            full_image = full_image.to(device, non_blocking=True)
            
            for i_polar, do_polar in enumerate([False, True]): 
                
                args.do_polar = do_polar
                fixation_map = data_transforms[do_polar](full_image)

                proba_label = get_batch(args, model[do_polar], fixation_map, 50)
                proba_label = proba_label[:, true_label].reshape(args.resolution)

                ax = axs[i_polar+2]
                
                display_heat(full_image_np, proba_label, args.resolution, model_name,  ax)
                
                ax.set_xticks([])
                ax.set_yticks([])
                
                if not do_polar:
                    args.do_polar = False
                    
                    fixation_map = data_transforms[False](full_image)
        
                    proba_label = get_batch(args, model_raw, fixation_map, 50)
                    proba_label = proba_label[:, true_label].reshape(args.resolution)
                    ax = axs[1]
                    display_heat(full_image_np, proba_label, args.resolution, model_name,  ax)
                    
                    

        all_axs[0,0].set_title('Image', fontsize=25)
        all_axs[0,1].set_title("No re-train", fontsize=25)
        all_axs[0,2].set_title("Cartesian", fontsize=25)
        all_axs[0,3].set_title("Retinotopic", fontsize=25)
        fig.set_facecolor(color='white')
        plt.show()
    


In [ ]:
# to_save(fig, name='fig-likelihood_map')